In [ ]:
# pip install adjustText 
# Install if needed: pip install adjustText

In [ ]:
# Import necessary libraries

import pandas as pd
from sklearn.metrics import pairwise_distances
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer, PowerTransformer
from sklearn.cluster import KMeans
from adjustText import adjust_text
from sklearn.decomposition import PCA

# This will ensure the outputs of the .transform() method are pandas data frames
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
# import csv file
spotify = pd.read_csv('../data/spotify_5000_songs.csv')

In [ ]:
spotify

In [ ]:
# create copy of df
spotify_df = spotify.copy()

In [ ]:
 # set song_name as index
spotify_df.columns = spotify_df.columns.str.strip()

In [ ]:
# set song_name as index

spotify_df = spotify_df.set_index(['name','artist'])

In [ ]:

# Drop the 'artist', 'id', and 'html' columns
spotify_df = spotify_df.drop(columns=['id', 'html','Unnamed: 0','type','mode','key','time_signature', 'duration_ms'],
                             errors='ignore'
                             )

In [ ]:
spotify_df

In [ ]:
spotify_df.describe().T

In [ ]:
spotify_df.hist(
    figsize=(15, 12),
    bins=30
)

plt.tight_layout()
plt.show()

In [ ]:
# PCA without scaling

pca_unscaled = PCA(n_components=2)
unscaled_pca = pca_unscaled.fit_transform(spotify_df)

# Single plot

plt.figure(figsize=(10, 7))
plt.scatter(
    unscaled_pca.iloc[:, 0],
    unscaled_pca.iloc[:, 1],
    s=8,
    alpha=0.5
)

plt.title('PCA Projection - Without Scaling')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

In [ ]:
# Refit PCA on current data with 9 features
pca_scaled = PCA(n_components=2)
scaled_pca = pca_scaled.fit_transform(spotify_df)

# Create loadings table
loadings = pd.DataFrame(
    pca_scaled.components_.T,
    columns=['PC1', 'PC2'],
    index=spotify_df.columns
)

print(loadings)

In [ ]:
# Create a MinMaxScaler object
scaler = MinMaxScaler()

# Scale the foods_df DataFrame
with_all_minmax_df = scaler.fit_transform(spotify_df)

In [ ]:
# Display age column of original DataFrame and age column of MinMaxed DataFrame to compare
pd.DataFrame({
    'original': spotify_df.iloc[:,-1],
    'min_max_scale': with_all_minmax_df.iloc[:,-1]
}).sort_values(by='original')

In [ ]:
with_all_minmax_df.describe().T

In [ ]:
# Stating that we want two plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Plotting the two plots
sns.histplot(data=spotify_df.iloc[:,-1], bins=10, kde=True, ax=ax1);
sns.histplot(with_all_minmax_df.iloc[:,-1], bins=10, kde=True, ax=ax2);

# Adding titles to the plots
ax1.set_title('Distribution of music features without scaling')
ax2.set_title('Distribution of music features with MinMax scaling')

plt.show()

In [ ]:
# PCA
pca_unscaled = PCA(n_components=2)
pca_minmax = PCA(n_components=2)

unscaled_pca = pca_unscaled.fit_transform(spotify_df)
minmax_pca = pca_minmax.fit_transform(with_all_minmax_df)

# If PCA output is a DataFrame, use .iloc
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax1.scatter(
    unscaled_pca.iloc[:, 0],
    unscaled_pca.iloc[:, 1],
    s=8,
    alpha=0.5
)

ax1.set_title('Without Scaling')
ax1.set_xlabel('Principal Component 1')
ax1.set_ylabel('Principal Component 2')

ax2.scatter(
    minmax_pca.iloc[:, 0],
    minmax_pca.iloc[:, 1],
    s=8,
    alpha=0.5
)

ax2.set_title('With MinMax Scaling')
ax2.set_xlabel('Principal Component 1')
ax2.set_ylabel('Principal Component 2')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

import seaborn as sns
import plotly.graph_objects as go


In [ ]:
# Set the maximum number of clusters to try
max_k = 100

# Create an empty list to store the inertia scores
inertia_list = []

# Iterate over the range of cluster numbers
for k in range(1, max_k+1):

    # Create a KMeans object with the specified number of clusters
    myKMeans = KMeans(n_clusters = k,
                      random_state = 200,
                      n_init=10)

    # Fit the KMeans model to the scaled data
    myKMeans.fit(with_all_minmax_df)

    # Append the inertia score to the list
    inertia_list.append(myKMeans.inertia_)

In [ ]:
# Set the Seaborn theme to darkgrid
sns.set_theme(style='darkgrid')

(
# Create a line plot of the inertia scores
sns.relplot(y = inertia_list,
            x = range(1, max_k + 1),
            kind = 'line',
            marker = 'o',
            height = 8,
            aspect = 2)
# Set the title of the plot
.set(title=f"Inertia score from 1 to {max_k} clusters")
# Set the axis labels
.set_axis_labels("Number of clusters", "Inertia score")
);

In [ ]:
# Set the maximum number of clusters to try
max_k = 100

# Create an empty list to store the silhouette scores
sil_scores = []

for k in range(2, max_k):

    # Create a KMeans object with the specified number of clusters
    kmeans = KMeans(n_clusters = k,
                    random_state = 200,
                    n_init=10)

    # Fit the KMeans model to the scaled data
    kmeans.fit(with_all_minmax_df)

    # Get the cluster labels
    labels = kmeans.labels_

    # Calculate the silhouette score
    sil_score = silhouette_score(with_all_minmax_df, labels)

    # Append the silhouette score to the list
    sil_scores.append(sil_score)

In [ ]:
(
sns.relplot(y = sil_scores,
            x = range(2, max_k),
            kind = 'line',
            marker = 'o',
            height = 8,
            aspect = 2)
.set(title=f"Silhouette score from 2 to {max_k - 1} clusters")
.set_axis_labels("Number of clusters", "Silhouette score")
);

In [ ]:
# Clean feature data
X_minmax = with_all_minmax_df.drop(
    columns=['cluster', 'main_cluster', 'sub_cluster'],
    errors='ignore'
).reset_index(drop=True)

# Create 8 main clusters
main_kmeans = KMeans(
    n_clusters=8,
    random_state=200,
    n_init=10
)

main_labels = main_kmeans.fit_predict(X_minmax)

# Create dataframe
main_clustered_df = X_minmax.copy()
main_clustered_df['main_cluster'] = main_labels

# Show sizes
main_clustered_df['main_cluster'].value_counts().sort_index()

In [ ]:

main_clustered_df['sub_cluster'] = -1
subcluster_results = {}

for main_cluster_id in sorted(main_clustered_df['main_cluster'].unique()):

    group = main_clustered_df[
        main_clustered_df['main_cluster'] == main_cluster_id
    ].drop(columns=['main_cluster', 'sub_cluster'])

    scores = {}

    # Wider search range
    for k in range(2, 26):

        if len(group) <= k:
            continue

        sub_kmeans = KMeans(
            n_clusters=k,
            random_state=200,
            n_init=10
        )

        sub_labels = sub_kmeans.fit_predict(group)
        scores[k] = silhouette_score(group, sub_labels)

    # Choose best k, but avoid overly simple splits
    # by ignoring k=2 and k=3 if the group is very large
    if len(group) > 600:
        filtered_scores = {k: v for k, v in scores.items() if k >= 4}
    else:
        filtered_scores = scores

    best_k = max(filtered_scores, key=filtered_scores.get)

    subcluster_results[main_cluster_id] = best_k

    print(f"Main cluster {main_cluster_id}: best subcluster k = {best_k}")

    final_sub_kmeans = KMeans(
        n_clusters=best_k,
        random_state=200,
        n_init=10
    )

    final_sub_labels = final_sub_kmeans.fit_predict(group)

    mask = main_clustered_df['main_cluster'] == main_cluster_id
    main_clustered_df.loc[mask, 'sub_cluster'] = final_sub_labels

main_clustered_df.groupby(
    ['main_cluster', 'sub_cluster']
).size().sort_values(ascending=False)

In [ ]:
# Show average audio-feature profile
# for each main cluster + subcluster

playlist_feature_profiles = main_clustered_df.groupby(
    ['main_cluster', 'sub_cluster']
).mean()

playlist_feature_profiles

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Feature data only
X = with_all_minmax_df.drop(
    columns=['cluster', 'main_cluster', 'sub_cluster'],
    errors='ignore'
).reset_index(drop=True)

# Cluster labels
cluster_labels = main_clustered_df['main_cluster'].reset_index(drop=True)

# t-SNE
tsne = TSNE(
    n_components=2,
    random_state=200,
    perplexity=30,
    learning_rate='auto',
    init='pca'
)

tsne_result = tsne.fit_transform(X)

# Create plot dataframe
tsne_df = pd.DataFrame({
    'tsne_1': tsne_result.iloc[:, 0],
    'tsne_2': tsne_result.iloc[:, 1],
    'cluster': cluster_labels
})

# Plot
plt.figure(figsize=(16, 10))

sns.scatterplot(
    data=tsne_df,
    x='tsne_1',
    y='tsne_2',
    hue='cluster',
    palette='tab10',
    s=45,
    alpha=0.7
)

plt.title('Final Spotify Cluster Map (5,235 Songs)', fontsize=18)
plt.xlabel('tsne_1')
plt.ylabel('tsne_2')
plt.legend(title='cluster')
plt.show()

In [ ]:
playlist_names = {
    (0, 0): "Fast Instrumental Intensity",
    (0, 1): "Dark Instrumental Drive",
    (0, 2): "Danceable Instrumental Energy",
    (0, 3): "Extreme Instrumental Intensity",
    (0, 4): "Heavy Instrumental Motion",
    (0, 5): "Moderate Instrumental Groove",
    (0, 6): "Low-Dance Instrumental Drive",
    (0, 7): "Live Instrumental Intensity",
    (0, 8): "Positive Instrumental Energy",

    (1, 0): "Dark High-Energy Vocals",
    (1, 1): "Dark Semi-Instrumental Energy",

    (2, 0): "Balanced Acoustic Groove",
    (2, 1): "Bright Acoustic Motion",
    (2, 2): "Warm Mid-Energy Groove",
    (2, 3): "Danceable Acoustic Instrumentals",
    (2, 4): "Bright Acoustic Energy",
    (2, 5): "Live Acoustic Feel-Good",
    (2, 6): "Soft Acoustic Warmth",

    (3, 0): "Ambient Instrumental Calm",
    (3, 1): "Soft Instrumental Atmospheres",

    (4, 0): "Live Rhythmic Energy",
    (4, 1): "Bright Live Groove",
    (4, 2): "Dark Live Energy",

    (5, 0): "Danceable Low-Mood Groove",
    (5, 1): "Moderate Energy Drive",
    (5, 2): "Dark Moderate Energy",
    (5, 3): "Mellow Rhythmic Drive",

    (6, 0): "Bright Danceable Groove",
    (6, 1): "Fast Bright Rhythms",
    (6, 2): "Danceable Positive Energy",
    (6, 3): "Bright Energy Drive",
    (6, 4): "Warm Danceable Energy",
    (6, 5): "High-Valence Dance Energy",
    (6, 6): "Instrumental Positive Groove",

    (7, 0): "Mellow Acoustic Vocals",
    (7, 1): "Soft Acoustic Low-Mood"
}

playlist_feature_profile = clustered_hierarchical.groupby(
    ['main_cluster', 'sub_cluster']
).mean()

playlist_feature_profile["playlist_name"] = [
    playlist_names.get(index, "Unclassified")
    for index in playlist_feature_profile.index
]

playlist_feature_profile

In [ ]:
# Create playlist dataframe from original data
playlist_final_df = spotify.copy()

# Clean column names
playlist_final_df.columns = playlist_final_df.columns.str.strip()

# Reset index to align with clustering dataframe
playlist_final_df = playlist_final_df.reset_index(drop=True)

# Add main and subcluster labels
playlist_final_df['main_cluster'] = main_clustered_df['main_cluster']
playlist_final_df['sub_cluster'] = main_clustered_df['sub_cluster']

# Create combined playlist ID
playlist_final_df['playlist_id'] = (
    playlist_final_df['main_cluster'].astype(str)
    + "_"
    + playlist_final_df['sub_cluster'].astype(str)
)

# Check result
playlist_final_df[['name', 'artist', 'main_cluster', 'sub_cluster', 'playlist_id']].head()

In [ ]:
# Display 3 sample songs for each playlist
# INCLUDING playlist names

for main_id in sorted(playlist_final_df['main_cluster'].unique()):
    
    print(f"\n\n================ MAIN CLUSTER {main_id} ================")
    
    subclusters = sorted(
        playlist_final_df[
            playlist_final_df['main_cluster'] == main_id
        ]['sub_cluster'].unique()
    )
    
    for sub_id in subclusters:
        
        # Get playlist name
        playlist_name = playlist_feature_profiles.loc[
            (main_id, sub_id),
            'playlist_name'
        ]
        
        # Sample songs
        sample_tracks = (
            playlist_final_df[
                (playlist_final_df['main_cluster'] == main_id) &
                (playlist_final_df['sub_cluster'] == sub_id)
            ][['name', 'artist']]
            .sample(
                n=min(
                    3,
                    len(
                        playlist_final_df[
                            (playlist_final_df['main_cluster'] == main_id) &
                            (playlist_final_df['sub_cluster'] == sub_id)
                        ]
                    )
                ),
                random_state=200
            )
        )
        
        print(f"\n----- Subcluster {sub_id} -----")
        print(f"Playlist Name: {playlist_name}")
        
        display(sample_tracks)